# Sorani (Kurdish) Voice Training — Piper TTS

Bu not defteri Sorani (ckb) sesini Piper'a eğitir. Veri: Mozilla Common Voice ckb. Eğitim `OHF-Voice/piper1-gpl`'in `piper.train` modülüyle yapılır.

Eğitim sonunda `model.onnx` + `model.onnx.json` üretilir; `deploy/stt/models/`'e koyup TTS'i devreye alırsın.

Adımlar: 0) GPU 1) sistem bağımlılıkları + piper kur 2) tar yükle 3) veri hazırla (CSV) 4) EĞİT 5) ONNX + indir

In [ ]:
# 0) GPU kontrol
!nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null || echo "GPU yok"

## 1) Sistem bağımlılıkları + Piper eğitim aracını kur
`piper-training` PyPI paketi YOK — Piper eğitimi resmi repo `OHF-Voice/piper1-gpl` üzerinden kurulur (`python3 -m piper.train`).

In [ ]:
# 1) Sistem paketleri + espeak-ng (fonem için)
!apt-get update -qq
!apt-get install -y -qq espeak-ng build-essential cmake ninja-build >> /tmp/apt.log 2>&1
print("apt tamam; espeak kurdish:", !espeak-ng --voices | grep -iE "ku|ckb|kur" | head -5)

In [ ]:
# 2) Piper1-gpl (eğitim dahil) clone + kur
import os
if not os.path.exists("piper1-gpl"):
    !git clone -q https://github.com/OHF-Voice/piper1-gpl.git
os.chdir("piper1-gpl")
!pip install -q -e '.[train]' 2>&1 | tail -5
print("piper import test:")
!python -c "import piper; print('piper ok')"
!python -m piper.train --help 2>&1 | head -3

## 2) Mozilla Common Voice ckb verisini yükle

In [ ]:
# 3) tar dosyasını yükle ve aç
from google.colab import files
import glob, os, tarfile
print("cv-corpus-27.0-...-ckb.tar dosyasını yükle:")
up = files.upload()
for fn in up:
    if fn.endswith((".tar.gz", ".tgz", ".tar")):
        os.makedirs("cv_extract", exist_ok=True)
        with tarfile.open(fn, "r:*") as t:
            t.extractall("cv_extract")
        print("açıldı:", fn)

ckb_dirs = glob.glob("cv_extract/**/ckb", recursive=True)
tsvs = glob.glob("cv_extract/**/*.tsv", recursive=True)
CV = ckb_dirs[0] if ckb_dirs else (os.path.dirname(tsvs[0]) if tsvs else None)
assert CV, "tsv bulunamadı"
print("CV:", CV)
print("tsv:", [os.path.basename(t) for t in glob.glob(os.path.join(CV,"*.tsv"))])
print("clips:", os.path.isdir(os.path.join(CV,"clips")))

## 3) Kadın Sorani konuşmacıyı seç + metin.csv hazırla

In [ ]:
# 4) Tüm tsv'leri birleştir → konuşmacı seç → audio klasörü + metin.csv
import os, glob, csv, shutil, random
librosa, soundfile, numpy = __import__("librosa", fromlist=["x"]), __import__("soundfile", fromlist=["x"]), __import__("numpy", fromlist=["x"])
from collections import Counter

CV = CV  # üstteki değer
os.makedirs("audio", exist_ok=True)

rows = []
for tsv in glob.glob(os.path.join(CV, "**", "*.tsv"), recursive=True):
    with open(tsv, "r", encoding="utf-8") as f:
        rd = csv.DictReader(f, delimiter="\t")
        for r in rd:
            try:
                up = int(r.get("up_votes",0) or 0); down = int(r.get("down_votes",0) or 0)
            except Exception: up, down = 1, 0
            if up > down:
                rows.append(r)

female = [r for r in rows if r.get("gender")=="female"]
use = female if len(female) >= 30 else rows
print("kullanılan:", len(use), "(kadın:", len(female), ")")

cnt = Counter(r["client_id"] for r in use)
top = cnt.most_common(1)[0][0]
sub = [r for r in use if r["client_id"] == top]
print("konuşmacı:", top, "kayıt:", len(sub))

target = 1200  # ~1 saat (her clip ~3sn)
sub = sub[:target]
clips_src = os.path.join(CV, "clips")

# Piper CSV: <wav_adı>|<metin>
with open("metin.csv", "w", encoding="utf-8") as fcsv:
    for i, r in enumerate(sub):
        src = os.path.join(clips_src, r["path"])
        if not os.path.exists(src): continue
        dst = f"audio/{i:05d}.wav"
        audio, sr = librosa.load(src, sr=22050, mono=True)
        soundfile.write(dst, audio, 22050)
        fcsv.write(f"{os.path.basename(dst)}|{r['sentence']}\n") 
print("hazır:", len(glob.glob("audio/*.wav")), "clips; metin.csv yazıldı")

## 4) Piper eğitimi

In [ ]:
# 5) Eğitim parametreleri
VOICE_NAME = "sorani-female"
ESPEAK_VOICE = "ku"      # espeak-ng Kürtçe sesi; ckb desteklenmiyorsa "ku" kullanılır
BATCH = 8                 # T4 16GB için küçük tut
print(VOICE_NAME, ESPEAK_VOICE)

In [ ]:
# 6) Çalıştır (Colab T4 GPU; ~1-3 saat)
import os
os.chdir("piper1-gpl") if os.path.isdir("piper1-gpl") and os.path.abspath(".").endswith("piper1-gpl") is False else None
# audio + metin.csv konumlarına göre
!python -m piper.train fit \  --data.voice_name "{VOICE_NAME}" \  --data.csv_path /content/metin.csv \  --data.audio_dir /content/audio \  --model.sample_rate 22050 \  --data.espeak_voice "{ESPEAK_VOICE}" \  --data.cache_dir /content/cache \  --data.config_path /content/config.json \  --data.batch_size {BATCH} \  --trainer.max_epochs 600 \  --trainer.accelerator gpu \  --trainer.devices 1
print("eğitim bitti")

## 5) ONNX'e dışa aktar + indir

In [ ]:
# 7) ONNX export
import glob
ckpt = sorted(glob.glob("/content/lightning_logs/version_*/checkpoints/*.ckpt"))[-1]
print("checkpoint:", ckpt)
!python -m piper.train.export_onnx --checkpoint "{ckpt}" --output-file /content/model.onnx
# config.json'u model.onnx.json olarak kopyala
!cp /content/config.json /content/model.onnx.json
!ls -la /content/model.onnx*

In [ ]:
# 8) İndir
from google.colab import files
import os
for f in ["/content/model.onnx", "/content/model.onnx.json"]:
    if os.path.exists(f):
        print("indir:", f)
        files.download(f)